In [1]:
from __future__ import annotations

import csv
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor,
    rationalize,
)

from config.ablation_config import ABLATION


RESULTS_PATH = Path(ABLATION.results_path)


MAX_VARIABLES = 10

RATIONAL_CROSS_RESIDUAL_ROUND_DECIMALS = 2

TRANSFORMATIONS = standard_transformations + (
    convert_xor,
    implicit_multiplication_application,
    rationalize,
)


@dataclass(frozen=True)
class SymbolicMatchResult:
    correct: bool
    successful_check: str
    error: str


def to_float(value: str) -> float:
    value = str(value).strip()

    if value == "":
        return np.nan

    return float(value)


def to_int(value: str) -> int:
    value = str(value).strip()

    if value == "":
        return -1

    return int(float(value))


def mean(values: list[float]) -> float:
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    return float(np.mean(arr))


def std(values: list[float]) -> float:
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]

    if arr.size <= 1:
        return 0.0

    return float(np.std(arr, ddof=1))


def format_mean_std(mu: float, sigma: float) -> str:
    if not np.isfinite(mu):
        return ""

    return f"{mu:.3e} ± {sigma:.1e}"


def format_float(value: float) -> str:
    if not np.isfinite(value):
        return ""

    return f"{value:.3e}"


def format_rate(value: float) -> str:
    if not np.isfinite(value):
        return ""

    return f"{value:.2f}"


def load_rows(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing ablation report: {path}")

    with path.open("r", newline="") as f:
        return list(csv.DictReader(f))


def base_locals() -> dict[str, object]:
    local_dict: dict[str, object] = {
        "sqrt": sp.sqrt,
        "log": sp.log,
        "ln": sp.log,
        "log10": lambda x: sp.log(x, 10),
        "exp": sp.exp,
        "sin": sp.sin,
        "cos": sp.cos,
        "tan": sp.tan,
        "Abs": sp.Abs,
        "abs": sp.Abs,
        "square": lambda x: x**2,
        "cube": lambda x: x**3,
        "pi": sp.pi,
        "E": sp.E,
    }

    for i in range(1, MAX_VARIABLES + 1):
        local_dict[f"x{i}"] = sp.Symbol(f"x{i}")

    local_dict["x"] = sp.Symbol("x1")

    return local_dict


def parse_symbolic_expression(expr_str: str) -> sp.Expr:
    expr_str = str(expr_str).strip()

    if expr_str == "":
        raise ValueError("empty expression")

    if expr_str.startswith("<symbolic extraction failed"):
        raise ValueError(expr_str)

    expr = parse_expr(
        expr_str,
        local_dict=base_locals(),
        transformations=TRANSFORMATIONS,
        evaluate=True,
    )

    if expr.has(sp.nan, sp.oo, -sp.oo, sp.zoo):
        raise ValueError(f"non-finite symbolic expression: {expr}")

    return expr


def round_numeric_constant_expr(expr: sp.Expr, decimals: int) -> sp.Expr:
    value = sp.N(expr, 50)

    if value.is_real is False:
        return expr

    rounded = round(float(value), decimals)

    if rounded == 0.0:
        return sp.Integer(0)

    return sp.Rational(str(rounded))


def round_numeric_atoms(expr: sp.Expr, decimals: int) -> sp.Expr:
    replacements = {}

    for atom in expr.atoms(sp.Number):
        if atom in (sp.pi, sp.E, sp.I):
            continue

        if atom.is_Integer:
            continue

        if atom.is_real is False:
            continue

        rounded = round(float(atom), decimals)

        if rounded == 0.0:
            replacements[atom] = sp.Integer(0)
        else:
            replacements[atom] = sp.Rational(str(rounded))

    return expr.xreplace(replacements)


def round_numeric_multiplicative_coefficients(
    expr: sp.Expr,
    decimals: int,
) -> sp.Expr:
    symbols = sorted(expr.free_symbols, key=lambda s: s.name)

    if not symbols:
        return round_numeric_constant_expr(expr, decimals)

    expr = sp.expand(expr)
    terms = sp.Add.make_args(expr)
    rounded_terms = []

    for term in terms:
        numeric_part, symbolic_part = term.as_independent(*symbols, as_Add=False)

        if numeric_part.is_number:
            rounded_numeric_part = round_numeric_constant_expr(
                numeric_part,
                decimals=decimals,
            )
            rounded_terms.append(rounded_numeric_part * symbolic_part)
        else:
            rounded_terms.append(term)

    return sp.expand(sp.Add(*rounded_terms))


def normalized_rational_components(expr: sp.Expr) -> tuple[sp.Expr, sp.Expr]:
    expr = sp.cancel(sp.together(expr))

    num, den = sp.fraction(expr)
    symbols = sorted(expr.free_symbols, key=lambda s: s.name)

    if not symbols:
        return sp.expand(num), sp.expand(den)

    try:
        den_poly = sp.Poly(den, *symbols)
    except Exception:
        return sp.expand(num), sp.expand(den)

    lc = den_poly.LC()

    if not lc.is_number:
        return sp.expand(num), sp.expand(den)

    lc_value = float(sp.N(lc, 50))

    if lc_value == 0.0:
        return sp.expand(num), sp.expand(den)

    num = sp.expand(num / lc)
    den = sp.expand(den / lc)

    return num, den


def rational_symbolic_match(found: sp.Expr, target: sp.Expr) -> SymbolicMatchResult:
    found_num, found_den = normalized_rational_components(found)
    target_num, target_den = normalized_rational_components(target)

    cross_residual = sp.expand(found_num * target_den - target_num * found_den)

    cross_residual = round_numeric_multiplicative_coefficients(
        cross_residual,
        decimals=RATIONAL_CROSS_RESIDUAL_ROUND_DECIMALS,
    )

    cross_residual = round_numeric_atoms(
        cross_residual,
        decimals=RATIONAL_CROSS_RESIDUAL_ROUND_DECIMALS,
    )

    cross_residual = sp.expand(cross_residual)
    cross_residual = sp.factor(cross_residual)
    cross_residual = sp.expand(cross_residual)

    rational_checks: list[tuple[str, Callable[[sp.Expr], sp.Expr]]] = [
        ("rational_cross_direct", lambda z: z),
        ("rational_cross_expand", sp.expand),
        ("rational_cross_factor", sp.factor),
        ("rational_cross_simplify", sp.simplify),
    ]

    for name, check in rational_checks:
        checked = check(cross_residual)

        if checked == 0:
            return SymbolicMatchResult(
                correct=True,
                successful_check=name,
                error="",
            )

    return SymbolicMatchResult(
        correct=False,
        successful_check="",
        error="",
    )


def recompute_symbolic_match(true_expr_str: str, found_expr_str: str) -> SymbolicMatchResult:
    try:
        target = parse_symbolic_expression(true_expr_str)
        found = parse_symbolic_expression(found_expr_str)

        return rational_symbolic_match(
            found=found,
            target=target,
        )

    except Exception as e:
        return SymbolicMatchResult(
            correct=False,
            successful_check="",
            error=str(e),
        )


def add_recomputed_symbolic_matches(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    updated_rows = []

    for row in rows:
        new_row = dict(row)

        true_expr = new_row.get("true_expr", "")
        found_expr = new_row.get("found_expr", "")

        match = recompute_symbolic_match(
            true_expr_str=true_expr,
            found_expr_str=found_expr,
        )

        new_row["rational_symbolic_match"] = str(int(match.correct))
        new_row["rational_symbolic_check"] = match.successful_check
        new_row["rational_symbolic_error"] = match.error

        updated_rows.append(new_row)

    return updated_rows


def modal_expression(rows: list[dict[str, str]]) -> tuple[str, int]:
    exprs = [row.get("found_expr", "").strip() for row in rows]
    counts = Counter(exprs)

    if not counts:
        return "", 0

    return counts.most_common(1)[0]


def aggregate_group(rows: list[dict[str, str]]) -> dict[str, object]:
    n_runs = len(rows)

    interp_mses = [to_float(row["test_interp_mse"]) for row in rows]
    extrap_mses = [to_float(row["test_extrap_mse"]) for row in rows]
    node_counts = [to_float(row["sympy_count_ops"]) for row in rows]
    matches = [to_int(row["rational_symbolic_match"]) for row in rows]

    modal_expr, modal_expr_count = modal_expression(rows)

    modal_rows = [
        row for row in rows
        if row.get("found_expr", "").strip() == modal_expr
    ]

    modal_extrap_mses = [
        to_float(row["test_extrap_mse"])
        for row in modal_rows
    ]

    recovery_rate = float(np.mean(matches)) if n_runs > 0 else np.nan

    return {
        "n_runs": n_runs,

        "interp_mse_mean": mean(interp_mses),
        "interp_mse_std": std(interp_mses),

        "extrap_mse_mean": mean(extrap_mses),
        "extrap_mse_std": std(extrap_mses),

        "modal_extrap_mse_mean": mean(modal_extrap_mses),
        "modal_expr_count": modal_expr_count,
        "modal_expr_fraction": modal_expr_count / n_runs if n_runs > 0 else np.nan,

        "node_count_mean": mean(node_counts),
        "node_count_std": std(node_counts),

        "recovery_rate": recovery_rate,

        "modal_found_expr": modal_expr,
    }


def group_rows(rows: list[dict[str, str]]) -> dict[tuple[str, str], list[dict[str, str]]]:
    grouped: dict[tuple[str, str], list[dict[str, str]]] = {}

    for row in rows:
        key = (
            row.get("ablation", "").strip(),
            row.get("group", "").strip(),
        )
        grouped.setdefault(key, []).append(row)

    return grouped


def print_summary(grouped: dict[tuple[str, str], list[dict[str, str]]]) -> None:
    print()
    print("Ablation summary")
    print("=" * 120)

    header = (
        f"{'ablation':32s} | "
        f"{'group':20s} | "
        f"{'interp mse':20s} | "
        f"{'extrap mse':20s} | "
        f"{'modal extrap':12s} | "
        f"{'nodes':16s} | "
        f"{'rec.':6s}"
    )

    print(header)
    print("-" * 120)

    for ablation, group in sorted(grouped.keys()):
        stats = aggregate_group(grouped[(ablation, group)])

        print(
            f"{ablation:32s} | "
            f"{group:20s} | "
            f"{format_mean_std(stats['interp_mse_mean'], stats['interp_mse_std']):20s} | "
            f"{format_mean_std(stats['extrap_mse_mean'], stats['extrap_mse_std']):20s} | "
            f"{format_float(stats['modal_extrap_mse_mean']):12s} | "
            f"{format_mean_std(stats['node_count_mean'], stats['node_count_std']):16s} | "
            f"{format_rate(stats['recovery_rate']):6s}"
        )


def main() -> None:
    rows = load_rows(RESULTS_PATH)
    rows = add_recomputed_symbolic_matches(rows)

    grouped = group_rows(rows)

    print_summary(grouped)


if __name__ == "__main__":
    main()


Ablation summary
ablation                         | group                | interp mse           | extrap mse           | modal extrap | nodes            | rec.  
------------------------------------------------------------------------------------------------------------------------
ceql_no_imaginary_penalty        | A1_three_variate_linear_rational | 2.870e+00 ± 6.4e+00  | 3.394e+00 ± 7.6e+00  | 1.957e-04    | 2.960e+01 ± 8.0e+00 | 0.00  
ceql_no_skip_connections         | A1_three_variate_linear_rational | 1.824e+01 ± 4.1e+01  | 3.587e+01 ± 8.0e+01  | 5.561e-09    | 4.040e+01 ± 1.2e+01 | 0.80  
ceql_original                    | A1_three_variate_linear_rational | 6.987e+00 ± 1.6e+01  | 3.460e+00 ± 7.7e+00  | 2.544e-07    | 2.300e+01 ± 9.7e+00 | 0.80  
ceql_original_with_log           | A1_three_variate_linear_rational | 2.317e+01 ± 1.9e+01  | 1.206e+01 ± 8.8e+00  | 5.425e+00    | 5.680e+01 ± 5.2e+01 | 0.20  
ceql_real_weights                | A1_three_variate_linear_rational | 3.536e